# Phase 3 – Rhetorical Role Prediction (RR)
## Experiment 4: Knowledge Distillation (MANDATORY) — Arch 4 Teacher → Arch 1 Student

**Technique**: **Knowledge Distillation (MANDATORY per specification)**  
**Teacher Model**: Phase-1 Architecture 4 – `ALiBiSwiGLULM` (~105M, Best Phase 1 Model PPL=23.66)  
**Student Model**: Phase-1 Architecture 1 – `DecoderLM` (~91M)  
**Teacher Checkpoint**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt`  
**Custom Tokenizer**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json`  
**Dataset**: Hugging Face `Exploration-Lab/CS-781-Capstone` (Phase_3_RR)  

---

### Key Highlights
- **Zip Archive Auto-Fix**: Eliminates `.format_version` loading errors for checkpoints.
- **Chunked Sentence Encoding**: Guarantees zero CUDA OOMs during both Teacher and Student passes.
- **Dual-Signal Distillation**: Combines Temperature-softened KL Divergence on logits with Cosine/MSE representation distillation.

## 1. Setup & Environment

In [ ]:
import os, gc, json, math, random, time, zipfile, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Paths & Hugging Face Dataset Download

In [ ]:
from huggingface_hub import login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")

TEACHER_MODEL_PATH = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
TOKENIZER_PATH     = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json'

# Student (Arch 1) fallback search or default
import glob
arch1_matches = glob.glob("/kaggle/input/**/arch1/**/decoder_lm.pt", recursive=True)
STUDENT_MODEL_PATH = arch1_matches[0] if arch1_matches else '/kaggle/input/phase1-arch1/decoder_lm.pt'

REPO = "Exploration-Lab/CS-781-Capstone"
print("Downloading Phase 3 RR datasets from Hugging Face...")
TRAIN_CL = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_CL   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_dev.json", repo_type="dataset", token=HF_TOKEN)
TRAIN_IT = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_IT   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_dev.json", repo_type="dataset", token=HF_TOKEN)
LABEL_VOCAB_PATH = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/label_vocab.json", repo_type="dataset", token=HF_TOKEN)
TEST_CL  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/CL_test.json", repo_type="dataset", token=HF_TOKEN)
TEST_IT  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/IT_test.json", repo_type="dataset", token=HF_TOKEN)

OUTPUT_DIR = '/kaggle/working/phase3_exp4_kd'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Distillation Hyperparameters
KD_TEMPERATURE = 4.0
ALPHA = 0.5   # Soft Logits Loss weight
BETA  = 0.2   # Representation Distillation weight
GAMMA = 0.3   # Hard Ground Truth Cross-Entropy weight

MAX_SEQ_LEN    = 256
CONTEXT_WINDOW = 3
CHUNK_SIZE     = 16
LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 0.01
EPOCHS         = 10
GRAD_CLIP      = 1.0

print("Knowledge Distillation hyperparameters configured.")

## 3. Checkpoint Loader with Zip Archive Auto-Fix

In [ ]:
def load_checkpoint_safe(model_path, prefix="arch"):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Checkpoint not found at: {model_path}")

    if os.path.isdir(model_path):
        files = os.listdir(model_path)
        pt_files = [f for f in files if f.endswith('.pt') or f.endswith('.pth')]
        if pt_files: model_path = os.path.join(model_path, pt_files[0])
        else: raise FileNotFoundError(f"No .pt file found in: {model_path}")

    try:
        return torch.load(model_path, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f"Direct load encountered exception ({str(e)[:80]}). Running zip auto-fix for {prefix}...")

    extract_dir = f"/kaggle/working/extracted_{prefix}_ckpt"
    fixed_pt    = f"/kaggle/working/fixed_{prefix}_decoder_lm.pt"
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(model_path, 'r') as z: z.extractall(extract_dir)
    nested = [f for f in os.listdir(extract_dir) if f.endswith('.pt') and f != os.path.basename(model_path)]
    if nested: return torch.load(os.path.join(extract_dir, nested[0]), map_location='cpu', weights_only=False)

    with zipfile.ZipFile(fixed_pt, 'w', compression=zipfile.ZIP_DEFLATED) as z_out:
        for root, _, files in os.walk(extract_dir):
            for file in files:
                if file in [".format_version", os.path.basename(fixed_pt)]: continue
                full_path = os.path.join(root, file)
                rel_path  = os.path.relpath(full_path, extract_dir)
                arcname   = os.path.join("archive", rel_path) if not rel_path.startswith("archive") else rel_path
                z_out.write(full_path, arcname=arcname)

    return torch.load(fixed_pt, map_location='cpu', weights_only=False)

t_ckpt = load_checkpoint_safe(TEACHER_MODEL_PATH, prefix="teacher")
print("Loaded Teacher checkpoint successfully.")

## 4. Custom ByteBPETokenizer

In [ ]:
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []; self.merge_rank = {}; self.itos = []; self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def encode(self, text, max_len=None):
        ids = [self.stoi.get(x, self.stoi['<unk>'])
               for w in text.split() for x in self._apply_merges_to_word(w)]
        return ids[:max_len] if max_len else ids

    def decode(self, ids):
        s = [self.itos[i] if i < len(self.itos) else '' for i in ids]
        words = [w for w in ''.join(s).split(END_OF_WORD) if w]
        return ' '.join(w.encode('latin-1').decode('utf-8', errors='replace') for w in words)

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f: d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']; t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
print(f"Tokenizer loaded. Vocab: {tokenizer.vocab_size}")

## 5. Dataset Loader

In [ ]:
def load_docs(path):
    with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
    if isinstance(data, list): return data
    elif isinstance(data, dict):
        if 'text' in data: return [data]
        docs = []
        for doc_id, body in data.items():
            if isinstance(body, list): docs.append({'id': doc_id, 'text': body})
            elif isinstance(body, dict):
                body['id'] = doc_id; docs.append(body)
        return docs
    return []

with open(LABEL_VOCAB_PATH, 'r', encoding='utf-8') as f: lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

cl_train = load_docs(TRAIN_CL); cl_dev = load_docs(DEV_CL)
it_train = load_docs(TRAIN_IT); it_dev = load_docs(DEV_IT)
cl_test  = load_docs(TEST_CL);  it_test= load_docs(TEST_IT)

all_lbls = [l for doc in cl_train + it_train for l in doc.get('labels', [])]
counts = Counter(all_lbls); total = sum(counts.values())
class_weights = torch.tensor(
    [(total / (NUM_LABELS * max(counts.get(l, 1), 1))) ** 0.5 for l in LABELS],
    dtype=torch.float32
).to(device)
class_weights = class_weights / class_weights.mean()
print(f"Loaded {NUM_LABELS} labels.")

## 6. Architecture 4 Teacher & Architecture 1 Student Models

In [ ]:
# ==================== ARCHITECTURE 4 (TEACHER) ====================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads):
    def gsp2(n):
        s = 2 ** (-(2 ** -(math.log2(n) - 3)))
        return [s * (s**i) for i in range(n)]
    if math.log2(n_heads) == int(math.log2(n_heads)):
        return torch.tensor(gsp2(n_heads))
    cp2 = 2 ** math.floor(math.log2(n_heads))
    return torch.tensor(gsp2(cp2) + gsp2(2*cp2)[0::2][:n_heads-cp2])

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.dh = d_model // n_heads; self.dropout = dropout
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.q_norm  = RMSNorm(self.dh); self.k_norm = RMSNorm(self.dh)
        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer('slopes', slopes)

    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k)
        pq = torch.arange(S, device=x.device).unsqueeze(1)
        pk = torch.arange(S, device=x.device).unsqueeze(0)
        alibi = self.slopes * (pk - pq).unsqueeze(0)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = alibi + causal.unsqueeze(0).unsqueeze(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1,2).contiguous().view(B, S, self.n_heads * self.dh))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class ALiBiBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n1   = RMSNorm(d_model)
        self.attn = ALiBiCausalAttention(d_model, n_heads, dropout)
        self.n2   = RMSNorm(d_model)
        self.ffn  = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x

@dataclass
class ALiBiConfig:
    vocab_size: int; d_model: int=768; n_layers: int=12; n_heads: int=12; d_ff: int=2560; dropout: float=0.1

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([ALiBiBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout) for _ in range(cfg.n_layers)])
        self.norm_f  = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks: x = b(x)
        return self.norm_f(x)

# ==================== ARCHITECTURE 1 (STUDENT) ====================
@dataclass
class LMConfig:
    vocab_size: int; d_model: int=768; n_layers: int=12; n_heads: int=12; d_ff: int=3072; max_seq_len: int=1024; dropout: float=0.1

def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    div = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    pe[:, 0::2] = torch.sin(pos/div); pe[:, 1::2] = torch.cos(pos/div)
    return pe

class StudentAttn(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads; self.dh = cfg.d_model // cfg.n_heads; self.scale = 1/math.sqrt(self.dh)
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.register_buffer('causal_mask', torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len, dtype=torch.bool)), persistent=False)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q_proj(x).view(B,T,self.n_heads,self.dh).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_heads,self.dh).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_heads,self.dh).transpose(1,2)
        s = (q@k.transpose(-2,-1))*self.scale
        s = s.masked_fill(~self.causal_mask[:T,:T], float('-inf'))
        a = F.softmax(s, dim=-1)
        z = (a@v).transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(z)

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model); self.attn = StudentAttn(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Dropout(cfg.dropout), nn.Linear(cfg.d_ff, cfg.d_model))
    def forward(self, x):
        x = x + self.attn(self.ln1(x)); x = x + self.ffn(self.ln2(x)); return x

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer('pos_emb', sinusoidal_pe(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:T]
        for b in self.blocks: x = b(x)
        return self.ln_f(x)

print("Architectures defined.")

## 7. Teacher & Student Classifiers with OOM-Safe Chunking

In [ ]:
class KDClassifier(nn.Module):
    def __init__(self, backbone, num_labels: int, is_teacher=False):
        super().__init__()
        self.backbone = backbone
        self.is_teacher = is_teacher
        d_model = backbone.cfg.d_model
        self.bilstm = nn.LSTM(d_model, d_model // 2, num_layers=1, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(d_model, num_labels)

    def get_sent_embs(self, input_ids, attention_mask, chunk_size=16):
        S = input_ids.size(0)
        sent_embs = []
        ctx = torch.no_grad() if self.is_teacher else torch.enable_grad()
        with ctx:
            for i in range(0, S, chunk_size):
                c_ids = input_ids[i : i + chunk_size]
                c_mask = attention_mask[i : i + chunk_size]
                h = self.backbone(c_ids)
                m = c_mask.unsqueeze(-1).float()
                sent_embs.append((h * m).sum(1) / m.sum(1).clamp(min=1e-9))
        return torch.cat(sent_embs, dim=0)

    def forward(self, input_ids, attention_mask, chunk_size=16, return_repr=False):
        embs = self.get_sent_embs(input_ids, attention_mask, chunk_size=chunk_size)
        x = embs.unsqueeze(0)
        lstm_out, _ = self.bilstm(x)
        lstm_out = lstm_out.squeeze(0)
        logits = self.classifier(lstm_out)
        if return_repr: return logits, lstm_out
        return logits

## 8. Initialize Models & Safe Loading

In [ ]:
# Teacher setup
t_raw = t_ckpt.get('config', {})
if hasattr(t_raw, '__dict__'): t_raw = t_raw.__dict__
td = t_raw if isinstance(t_raw, dict) else {}
t_cfg = ALiBiConfig(
    vocab_size=t_ckpt.get('vocab_size', tokenizer.vocab_size),
    d_model=td.get('d_model', 768), n_layers=td.get('n_layers', 12),
    n_heads=td.get('n_heads', 12), d_ff=td.get('d_ff', 2560)
)
t_backbone = ALiBiSwiGLULM(t_cfg)
t_backbone.load_state_dict(t_ckpt.get('model_state_dict', t_ckpt), strict=False)
for p in t_backbone.parameters(): p.requires_grad = False
teacher = KDClassifier(t_backbone, NUM_LABELS, is_teacher=True).to(device)

# Student setup
s_cfg = LMConfig(vocab_size=tokenizer.vocab_size)
s_backbone = DecoderLM(s_cfg)
s_ckpt = None
if os.path.exists(STUDENT_MODEL_PATH):
    try:
        s_ckpt = load_checkpoint_safe(STUDENT_MODEL_PATH, prefix="student")
        s_backbone.load_state_dict(s_ckpt.get('model_state_dict', s_ckpt), strict=False)
        print("Loaded pretrained Student weights.")
    except Exception as e:
        print(f"Student initialized from scratch ({e}).")

student = KDClassifier(s_backbone, NUM_LABELS, is_teacher=False).to(device)
print("Teacher and Student models ready.")

## 9. Dataset with Context Window

In [ ]:
class RRDataset(Dataset):
    def __init__(self, docs, tokenizer, label2id, max_seq_len, context_window=3, domain_prefix='', is_test=False):
        self.items = []
        pad_id = tokenizer.stoi.get('<pad>', 0)
        for doc in docs:
            sents = doc['text']
            labels = doc.get('labels', [None] * len(sents))
            ids_list, masks_list, lbls_list = [], [], []
            for i, (sent, lbl) in enumerate(zip(sents, labels)):
                s = max(0, i - context_window)
                e = min(len(sents), i + context_window + 1)
                ctx = f"{domain_prefix} " + " </s> ".join(sents[s:i])
                ctx += f" </s> {sent} </s> " + " </s> ".join(sents[i+1:e])
                ids = tokenizer.encode(ctx.strip(), max_len=max_seq_len)
                pad = max_seq_len - len(ids)
                ids_list.append(ids + [pad_id] * pad)
                masks_list.append([1] * len(ids) + [0] * pad)
                lbls_list.append(label2id.get(lbl, 0) if (not is_test and lbl) else -1)
            self.items.append({
                'doc_id': doc['id'],
                'input_ids': torch.tensor(ids_list, dtype=torch.long),
                'attention_mask': torch.tensor(masks_list, dtype=torch.long),
                'labels': torch.tensor(lbls_list, dtype=torch.long),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

collate_fn = lambda b: b

from torch.utils.data import ConcatDataset
train_ds = ConcatDataset([
    RRDataset(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]"),
    RRDataset(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
])
dev_ds = ConcatDataset([
    RRDataset(cl_dev, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]"),
    RRDataset(it_dev, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
])
cl_te = RRDataset(cl_test, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]", is_test=True)
it_te = RRDataset(it_test, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]", is_test=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=1, shuffle=False, collate_fn=collate_fn)
print(f"Train docs: {len(train_ds)} | Dev docs: {len(dev_ds)}")

## 10. Two-Stage Distillation Training

In [ ]:
# Stage 1: Fine-tune Teacher head quickly (4 epochs)
print("=== Phase 1: Fine-tuning Teacher Head ===")
teacher_opt = AdamW([p for p in teacher.parameters() if p.requires_grad], lr=2e-4)
for ep in range(4):
    teacher.train(); ep_loss = 0.0; steps = 0
    for batch in train_loader:
        doc = batch[0]
        ids, mask, lbls = doc['input_ids'].to(device), doc['attention_mask'].to(device), doc['labels'].to(device)
        vm = lbls >= 0
        if vm.sum() == 0: continue
        with torch.amp.autocast('cuda'):
            logits = teacher(ids, mask, chunk_size=CHUNK_SIZE)
            loss = F.cross_entropy(logits[vm], lbls[vm], weight=class_weights)
        teacher_opt.zero_grad(); loss.backward(); teacher_opt.step()
        ep_loss += loss.item(); steps += 1
    print(f"Teacher Epoch {ep+1}/4 | Loss: {ep_loss/max(1,steps):.4f}")

for p in teacher.parameters(): p.requires_grad = False
teacher.eval()

# Stage 2: Distill Teacher into Student
def kd_loss_fn(s_logits, t_logits, T=4.0):
    p_s = F.log_softmax(s_logits / T, dim=-1)
    p_t = F.softmax(t_logits / T, dim=-1)
    return F.kl_div(p_s, p_t, reduction='batchmean') * (T * T)

student_opt = AdamW([p for p in student.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(student_opt, T_max=EPOCHS * len(train_loader), eta_min=1e-6)
best_f1 = 0.0
best_ckpt = os.path.join(OUTPUT_DIR, 'best_student_kd.pt')

print("\n=== Phase 2: Distilling to Student ===")
for epoch in range(EPOCHS):
    student.train(); ep_loss = 0.0; steps = 0
    for batch in train_loader:
        doc = batch[0]
        ids, mask, lbls = doc['input_ids'].to(device), doc['attention_mask'].to(device), doc['labels'].to(device)
        vm = lbls >= 0
        if vm.sum() == 0: continue

        with torch.no_grad():
            t_logits, t_rep = teacher(ids, mask, chunk_size=CHUNK_SIZE, return_repr=True)

        with torch.amp.autocast('cuda'):
            s_logits, s_rep = student(ids, mask, chunk_size=CHUNK_SIZE, return_repr=True)
            loss_kd = kd_loss_fn(s_logits, t_logits, KD_TEMPERATURE)
            loss_rep = F.mse_loss(s_rep, t_rep)
            loss_ce = F.cross_entropy(s_logits[vm], lbls[vm], weight=class_weights)
            loss = ALPHA * loss_kd + BETA * loss_rep + GAMMA * loss_ce

        student_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in student.parameters() if p.requires_grad], GRAD_CLIP)
        student_opt.step(); scheduler.step()
        ep_loss += loss.item(); steps += 1

    # Evaluate student
    student.eval(); all_p, all_l = [], []
    with torch.no_grad():
        for batch in dev_loader:
            doc = batch[0]
            ids, mask, lbls = doc['input_ids'].to(device), doc['attention_mask'].to(device), doc['labels'].to(device)
            with torch.amp.autocast('cuda'): logits = student(ids, mask, chunk_size=CHUNK_SIZE)
            all_p.extend(logits.argmax(-1).cpu().tolist())
            all_l.extend(lbls.cpu().tolist())
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    f1 = f1_score([all_l[i] for i in vm], [all_p[i] for i in vm], average='macro', zero_division=0)
    print(f"Distill Epoch {epoch+1}/{EPOCHS} | Loss: {ep_loss/max(1,steps):.4f} | Student Dev Macro-F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({'model_state_dict': student.state_dict(), 'f1': f1}, best_ckpt)
        print(f"  -> Best student saved (F1: {best_f1:.4f})")

print(f"Peak Student Dev Macro-F1: {best_f1:.4f}")

## 11. Predictions & Codabench Submission

In [ ]:
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
student.load_state_dict(ckpt['model_state_dict'])
student.eval()

def gen_preds(model, test_ds):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    preds = {}
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids, mask = doc['input_ids'].to(device), doc['attention_mask'].to(device)
            with torch.amp.autocast('cuda'):
                logits = model(ids, mask, chunk_size=CHUNK_SIZE)
            pred_ids = logits.argmax(-1).cpu().tolist()
            preds[doc['doc_id']] = [ID2LABEL[p] for p in pred_ids]
    return preds

cl_preds = gen_preds(student, cl_te)
it_preds = gen_preds(student, it_te)
all_preds = {**cl_preds, **it_preds}

pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(all_preds, f, indent=2, ensure_ascii=False)

zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname='predictions.json')
print(f"Submission ready at: {zip_path}")